In [ ]:
# Install dependencies
!pip install -U datasets huggingface_hub fsspec transformers tokenizers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.3/515.3 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 74.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.33.2
    Uninstalling huggingface-hub-0.33.2:
      Successfully uninstalled huggingface-hub-0.33.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.1
    Uninstalling transformers-4.53.1:
      Successfully uninstalled transformers-4.53.1
  Attempting uninstall: datasets
    Found existing installation: d

In [ ]:
from datasets import load_dataset
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/733k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/6.36M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [ ]:
print(dataset['train'][:5])
print(dataset['train'][:5]['text'])
print(dataset)

{'text': ['', ' = Valkyria Chronicles III = \n', '', ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n', " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game m

In [ ]:
# Clean dataset
'''
Replace with this if no need to count the cleaned rows
def clean_text(example):
    example['text'] = example['text'].strip().replace('\n', ' ')
    return example
'''
# Counter for cleaned rows
clean_count = 0

# Define cleaning function with counter
def clean_text_and_count(example):
    global clean_count
    original = example['text']
    cleaned = original.strip().replace('\n', ' ')
    if original != cleaned:
        clean_count += 1
    example['text'] = cleaned
    return example

# Apply cleaning
dataset = dataset.map(clean_text_and_count)

# Print a few rows to verify
print(dataset['train'][:5])

# Show count of cleaned rows
print(f"\nNumber of rows cleaned in train set: {clean_count}")

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

{'text': ['', '= Valkyria Chronicles III =', '', 'Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " .', "The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgi

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
from transformers import PreTrainedTokenizerFast

# Step 1: Build and configure tokenizer
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFKC(),
    normalizers.Lowercase()
])

# Step 2: Train tokenizer on dataset
trainer = trainers.BpeTrainer(
    vocab_size=50000,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)
tokenizer.train_from_iterator(dataset["train"]["text"], trainer=trainer)

# Step 3: Wrap directly with PreTrainedTokenizerFast
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)


In [ ]:
#Checking the created tokens
tokens = hf_tokenizer.tokenize("Valkyria Chronicles III")
print(tokens)

In [ ]:
def tokenize_function(examples):
    return hf_tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    )
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

In [ ]:
#Checking the output
# First 2 tokenized rows
for i in range(5):
    input_ids = tokenized_dataset["train"][i]["input_ids"]
    tokens = hf_tokenizer.convert_ids_to_tokens(input_ids)
    print(f"Sample {i+1}")
    print("Tokens:", tokens)
    print("Decoded:", hf_tokenizer.decode(input_ids, skip_special_tokens=True))


Sample 1
Tokens: ['[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '

In [ ]:
# Configure model
import torch
from transformers import GPT2Config, GPT2LMHeadModel
config = GPT2Config(
    vocab_size=50000,
    n_embd=768,
    n_layer=6,
    n_head=12
)
model = GPT2LMHeadModel(config)
'''
for tuning do below, right now it's training on Wikitext-2-raw-v1 dataset. This is GPT2-style decoder-only causal language model (CLM). This model is learning to predict the next token, given previous tokens (i.e., causal language modeling). It will not tune it.
model = GPT2LMHeadModel.from_pretrained("gpt2")
hf_tokenizer = AutoTokenizer.from_pretrained("gpt2")
'''


'\nfor tuning do below, right now it\'s training on Wikitext-2-raw-v1 dataset. This is GPT2-style decoder-only causal language model (CLM). This model is learning to predict the next token, given previous tokens (i.e., causal language modeling). It will not tune it.\nmodel = GPT2LMHeadModel.from_pretrained("gpt2")\nhf_tokenizer = AutoTokenizer.from_pretrained("gpt2")\n'

In [ ]:
# Setup training
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer
data_collator = DataCollatorForLanguageModeling(
    tokenizer=hf_tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./llm_output",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    save_steps=500,
    logging_steps=100,
    report_to="none"  # Disables wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"].select(range(10000)),
    tokenizer=hf_tokenizer
)

/tmp/ipython-input-11-1408021568.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Start training
trainer.train()

# Save model
model.save_pretrained("custom_llm")
hf_tokenizer.save_pretrained("custom_llm")

In [ ]:
from transformers import pipeline

# Load your trained model and tokenizer
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast

model = GPT2LMHeadModel.from_pretrained("custom_llm")
tokenizer = PreTrainedTokenizerFast.from_pretrained("custom_llm")

# Setup generation pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Try a prompt
prompt = "Artificial intelligence is"
outputs = generator(prompt, max_length=50, do_sample=True)

print(outputs[0]["generated_text"])
'''Output is-- (clearly shows the data we trained is not for this task)
Artificial intelligence is a great long, many of the name of the song is a way of the most of the north national film. the film is " is a new church of the film song was the first national world war of the main episode " a island of the original world war, the " early end of the second number of the new year of the main war galveston has in the united kingdom of the first song of the " the united states is ", the end of the first of the national second second united states of other original " is the early game of the first island century ii is a second united states for the first song is also also has been called the united states the last end of the end of a result of the first part of the world war is used a second @-@ century. in the song's " i is a " in " the second catechism, is a song of the film is an first of the great early song of the united states, and the song " the most of the end of " is a " a new music film's album first first original " i first church of the catechism song of the number of the united states is a new album " ode of ireland's first most important first early " the song ", the first season of the

'''

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Artificial intelligence is a great long, many of the name of the song is a way of the most of the north national film. the film is " is a new church of the film song was the first national world war of the main episode " a island of the original world war, the " early end of the second number of the new year of the main war galveston has in the united kingdom of the first song of the " the united states is ", the end of the first of the national second second united states of other original " is the early game of the first island century ii is a second united states for the first song is also also has been called the united states the last end of the end of a result of the first part of the world war is used a second @-@ century. in the song's " i is a " in " the second catechism, is a song of the film is an first of the great early song of the united states, and the song " the most of the end of " is a " a new music film's album first first original " i first church of the catechism s